# Lab 1: Hello Agent — Your First Claude API Call

**Module 1: Foundations of Agentic AI**  
**Duration:** 30 minutes  
**Difficulty:** Beginner  

---

## What You'll Learn

By the end of this lab, you will:
1. Set up the Anthropic Python SDK and authenticate with your API key
2. Understand message roles (`user`, `assistant`, `system`) and how they shape agent behavior
3. Make your first Claude API call and parse the response
4. Control agent behavior with system prompts
5. Build a simple conversational agent with memory (message history)

## How This Connects to the Agent Loop

```mermaid
flowchart TD
    A["PERCEIVE\nYou send a message (user role)"] --> B["REASON\nClaude thinks (guided by system prompt)"]
    B --> C["ACT\nClaude responds (assistant role)"]
    style A fill:#bee3f8,color:#1a202c
    style B fill:#c4b5fd,color:#1a202c
    style C fill:#fed7aa,color:#1a202c,stroke:#dd6b20,stroke-width:3px
```

> **This lab focuses on the ACT step** — the LLM brain. In Labs 2 and 3, we add tools and multi-step reasoning.

In this lab we focus on the **brain** of the agent — the LLM. In Labs 2 and 3, we'll add tools and multi-step reasoning.

---
## Step 1: Install the Anthropic SDK

Run this cell to install the official Anthropic Python library. If you're on Google Colab, this is all you need.

In [ ]:
!pip install -q anthropic

## Step 2: Set Up Your API Key

You need a Claude API key from [console.anthropic.com](https://console.anthropic.com/). 

**Security best practice:** Never hardcode API keys in notebooks. We'll use `getpass` so your key stays hidden.

In [ ]:
import os
from getpass import getpass

# Prompt for API key (won't display on screen)
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")

print("✓ API key set successfully")

## Step 3: Initialize the Client

The `Anthropic` client automatically reads `ANTHROPIC_API_KEY` from your environment. One line of code — you're connected.

In [ ]:
import anthropic

client = anthropic.Anthropic()

# We'll use Sonnet for all labs — fast, capable, cost-effective for learning
MODEL = "claude-sonnet-4-5-20241022"

print(f"✓ Client initialized. Using model: {MODEL}")

---
## Step 4: Your First API Call

The Claude API uses a **messages** interface. Every conversation is a list of messages with roles:

| Role | Who's Talking | Purpose |
|------|--------------|----------|
| `user` | You (the human) | Asks questions, gives instructions |
| `assistant` | Claude (the AI) | Responds, reasons, takes actions |
| `system` | The developer | Sets behavior rules (invisible to end users) |

Let's make our first call:

In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=256,
    messages=[
        {"role": "user", "content": "What is an AI agent in one sentence?"}
    ]
)

print(response.content[0].text)

### Understanding the Response Object

The API returns a rich response object, not just text. Let's explore it:

In [ ]:
print("=== Response Anatomy ===")
print(f"Model used:    {response.model}")
print(f"Stop reason:   {response.stop_reason}")
print(f"Input tokens:  {response.usage.input_tokens}")
print(f"Output tokens: {response.usage.output_tokens}")
print(f"Content type:  {response.content[0].type}")
print(f"Content text:  {response.content[0].text[:100]}...")

**Key fields to know:**
- `stop_reason`: Why the model stopped — `"end_turn"` means it finished naturally, `"max_tokens"` means it was cut off, `"tool_use"` means it wants to use a tool (Lab 2!)
- `usage`: Token counts for billing — input tokens (your prompt) + output tokens (Claude's response)
- `content`: A list of content blocks. Usually one text block, but can include tool_use blocks (Lab 2)

---
## Step 5: System Prompts — Shaping Agent Behavior

The **system prompt** is the most powerful lever you have. It defines who the agent IS — its personality, expertise, constraints, and rules.

Think of it as the agent's **job description**. A chatbot says "I'm a helpful assistant." An agent says "I'm a senior financial analyst who uses these specific tools to produce quarterly reports."

In [ ]:
# Without a system prompt — generic response
generic_response = client.messages.create(
    model=MODEL,
    max_tokens=256,
    messages=[
        {"role": "user", "content": "Analyze the risks of investing in Tesla stock."}
    ]
)

print("=== Without System Prompt ===")
print(generic_response.content[0].text[:500])
print("\n" + "="*50 + "\n")

# With a system prompt — focused, expert response
expert_response = client.messages.create(
    model=MODEL,
    max_tokens=256,
    system="You are a senior equity research analyst at Goldman Sachs with 15 years of experience. "
           "You analyze stocks using fundamental analysis, focusing on P/E ratios, revenue growth, "
           "competitive moats, and management quality. Be specific and quantitative. "
           "Always include a risk rating: LOW, MEDIUM, or HIGH.",
    messages=[
        {"role": "user", "content": "Analyze the risks of investing in Tesla stock."}
    ]
)

print("=== With Expert System Prompt ===")
print(expert_response.content[0].text[:500])

**Notice the difference?** The system prompt transforms a generic chatbot into a domain expert. This is the first step toward building an agent — giving it a specific identity and expertise.

---
## Step 6: Conversation Memory — Multi-Turn Dialogue

A single API call has no memory. Claude doesn't remember what you said before. To create a conversation, **you** manage the message history.

This is a critical concept: **the LLM is stateless**. Memory is YOUR responsibility.

In [ ]:
def chat(user_message: str, conversation_history: list[dict], system_prompt: str = "") -> str:
    """Send a message and get a response, maintaining conversation history.
    
    Args:
        user_message: The user's input text.
        conversation_history: List of prior messages (modified in place).
        system_prompt: Optional system prompt to shape behavior.
    
    Returns:
        The assistant's response text.
    """
    # Add the user's message to history
    conversation_history.append({"role": "user", "content": user_message})
    
    # Make the API call with FULL history
    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=system_prompt,
        messages=conversation_history
    )
    
    assistant_message = response.content[0].text
    
    # Add the assistant's response to history
    conversation_history.append({"role": "assistant", "content": assistant_message})
    
    return assistant_message

In [ ]:
# Start a conversation
history = []
system = "You are a helpful AI tutor teaching someone about agentic AI. Be concise."

# Turn 1
reply = chat("What's the difference between a chatbot and an AI agent?", history, system)
print(f"Turn 1: {reply}\n")

# Turn 2 — Claude remembers Turn 1
reply = chat("Can you give me a real-world example of each?", history, system)
print(f"Turn 2: {reply}\n")

# Turn 3 — Claude remembers Turns 1 AND 2
reply = chat("Which one would you use for automating expense reports?", history, system)
print(f"Turn 3: {reply}")

In [ ]:
# Let's peek at the conversation history
print(f"Messages in history: {len(history)}\n")
for i, msg in enumerate(history):
    role = msg['role'].upper()
    text = msg['content'][:80] + "..." if len(msg['content']) > 80 else msg['content']
    print(f"  [{i}] {role}: {text}")

**Key insight:** Every API call sends the ENTIRE conversation history. This is how Claude "remembers" — you replay the whole conversation each time. This is **conversation memory**, the simplest form of agent memory.

---
## Step 7: Structured Output — Getting Useful Data from Claude

Agents don't just chat — they produce structured output that other systems can consume. Let's ask Claude to return JSON.

In [ ]:
import json

response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    system="You are a data extraction agent. Always respond with valid JSON only — no markdown, no explanation.",
    messages=[{
        "role": "user",
        "content": """Extract the following information from this text and return as JSON:

Text: "John Smith is a 35-year-old software engineer at Google in Mountain View, CA. 
He has 10 years of experience in Python and machine learning. His email is john@example.com."

Extract: name, age, job_title, company, location, years_experience, skills (list), email"""
    }]
)

# Parse the JSON response
raw_text = response.content[0].text
data = json.loads(raw_text)

print("Extracted data:")
print(json.dumps(data, indent=2))
print(f"\nName: {data['name']}")
print(f"Skills: {', '.join(data['skills'])}")

**Why this matters for agents:** When an agent uses tools, it generates structured JSON to specify which tool to call and with what parameters. This is the same pattern — asking the LLM to produce machine-readable output.

---
## Step 8: Putting It Together — A Simple Q&A Agent

Let's combine everything into a reusable agent class. This is a **brain-only** agent (no tools yet — that's Lab 2), but it demonstrates the pattern.

In [ ]:
class SimpleAgent:
    """A basic conversational agent with memory and a configurable persona."""
    
    def __init__(self, system_prompt: str, model: str = MODEL):
        """Initialize the agent with a system prompt and model.
        
        Args:
            system_prompt: Defines the agent's identity and behavior.
            model: The Claude model to use.
        """
        self.client = anthropic.Anthropic()
        self.model = model
        self.system_prompt = system_prompt
        self.conversation_history: list[dict] = []
    
    def send(self, message: str) -> str:
        """Send a message and get a response.
        
        Args:
            message: The user's input.
        
        Returns:
            The agent's response text.
        """
        self.conversation_history.append({"role": "user", "content": message})
        
        response = self.client.messages.create(
            model=self.model,
            max_tokens=1024,
            system=self.system_prompt,
            messages=self.conversation_history
        )
        
        reply = response.content[0].text
        self.conversation_history.append({"role": "assistant", "content": reply})
        return reply
    
    def reset(self) -> None:
        """Clear conversation history to start fresh."""
        self.conversation_history = []
        print("✓ Conversation history cleared.")


# Create a specialized agent
agent = SimpleAgent(
    system_prompt=(
        "You are an Agentic AI tutor. You explain concepts using analogies and examples. "
        "Keep responses under 3 sentences unless asked for detail. "
        "End each response with a follow-up question to check understanding."
    )
)

# Have a conversation
print(agent.send("What is the agent loop?"))
print("\n---\n")
print(agent.send("The perceive-reason-act-observe cycle?"))
print("\n---\n")
print(agent.send("What happens if the agent gets stuck in the loop?"))

---
## Exercises

### Exercise 1: Create a Specialized Agent
Create a `SimpleAgent` with a system prompt that makes it act as a **code reviewer**. It should:
- Analyze code for bugs and style issues
- Rate code quality from 1-10
- Suggest specific improvements

Test it with a Python function.

In [ ]:
# YOUR CODE HERE
# Hint: Create a SimpleAgent with a code-reviewer system prompt,
# then send it a code snippet to review.



### Exercise 2: Token Cost Calculator
Write a function that takes a response object and calculates the estimated cost in USD.  
**Pricing (Claude Sonnet):** $3.00 per 1M input tokens, $15.00 per 1M output tokens.

In [ ]:
# YOUR CODE HERE
def calculate_cost(response) -> dict:
    """Calculate the cost of a Claude API response.
    
    Args:
        response: The API response object.
    
    Returns:
        Dict with input_cost, output_cost, and total_cost in USD.
    """
    pass  # Implement this



### Exercise 3: Conversation Summary
After a multi-turn conversation, ask Claude to summarize the conversation so far. This is a primitive form of **memory compression** — a real technique used in production agents when conversations get too long.

In [ ]:
# YOUR CODE HERE
# Hint: Use the `history` variable from Step 6, add a message asking for a summary.



---
## Key Takeaways

1. **The Anthropic SDK is simple** — `client.messages.create()` is your main function
2. **Message roles matter** — `system` shapes behavior, `user` provides input, `assistant` is the response
3. **Memory is your responsibility** — Claude is stateless; you manage the conversation history
4. **System prompts are the first lever** — Before adding tools, the system prompt defines what the agent IS
5. **Structured output enables automation** — JSON responses let agents feed data to other systems

## What's Next

In **Lab 2**, we'll give our agent **tools** — the ability to interact with the real world. The agent will decide WHICH tool to use, call it, observe the result, and decide what to do next. That's where it goes from "chatbot" to "agent."

---
*Lab 1 Complete — Module 1: Foundations of Agentic AI*